In [1]:
from bokeh.plotting import figure
from bokeh.models import Span
from bokeh.io import show, output_notebook
output_notebook()
import numpy as np
import biocircuits

import scipy

from typing import Callable

Loading BokehJS ...

# Figure 1

## B

In [14]:
# Pulse
def pulse(t, t0, t1, J):
    return J * (t >= t0) * (t <= t1) / (t1 - t0)

# Parameters
t0 = 0 # seconds
t1 = 0.2 # seconds
J = 1.0 # amplitude * seconds

# Time Spacing
t = np.linspace(-0.4, 15, 1000)

# Compute the pulse
input = pulse(t, t0, t1, J)

# Plot the pulse
input_plot = figure(width=400, height=100, y_axis_label = 'Input', x_axis_label = 'Time (s)')
input_plot.yaxis.major_tick_line_color = None
input_plot.yaxis.minor_tick_line_color = None
input_plot.yaxis.major_label_text_font_size = '0pt'
input_plot.line(t, input, line_width=2)
show(input_plot)

In [15]:
# Basis functions
def basis_function(t, tau, n):
    return (t>=0) * ((t/tau)**n) * np.exp(-t/tau) / scipy.special.factorial(n)

# Parameters
tau = 0.1 # seconds

# Plot a few basis functions: 1, 8, 15, .., 99
basis_plot = figure(width=400, height=300, y_axis_label = r'Basis Functions \[g_n\]', x_axis_label = 'Time (s)')
for n in range(1, 100, 7):
    g_n = basis_function(t, tau, n)
    basis_plot.line(t, J * g_n, line_width=2)

show(basis_plot)

In [30]:
# Weighted sum of basis functions
def weighted_sum(t, tau, weights):
    return sum(w * basis_function(t, tau, n) for n, w in enumerate(weights))

# uniform weights
uniform_weights = J * np.ones(100)

# Optimal weights
optimization_times = np.linspace(0, 10, 1000)
def objective_function(s):
    output = weighted_sum(optimization_times, tau, s)
    return np.sum((output - 1)**2)
optimal_weights = scipy.optimize.minimize(objective_function, uniform_weights, bounds=[(-5, 5) for _ in range(len(uniform_weights))]).x
print("Optimal weights:", optimal_weights)

# Compute the weighted sum
uniform_output = weighted_sum(t, tau, uniform_weights)
optimal_output = weighted_sum(t, tau, optimal_weights)

# Plot the weighted sum
output_plot = figure(width=400, height=300, y_axis_label = r'Output \[r_{out}\]', x_axis_label = 'Time (s)')
output_plot.line(t, uniform_output, line_width=2, legend_label='Uniform Weights', color='blue')
output_plot.line(t, optimal_output, line_width=2, legend_label='Optimal Weights', color='green')
output_plot.line(np.linspace(0,10,10), np.ones(10), line_width=2, color='red', line_dash='dashed')

output_plot.legend.location = "bottom_left"
show(output_plot)

Optimal weights: [ 0.99035536  1.10964595  0.67912613  1.27647153  1.15590022  0.81532471
  0.79161889  1.01432622  1.18427392  1.15467145  0.99548048  0.86021551
  0.84593867  0.94303956  1.07010712  1.14160355  1.11904804  1.02343187
  0.91367338  0.85060096  0.86680165  0.95450869  1.07305937  1.1691824
  1.1998969   1.14893149  1.03158115  0.8878544   0.76766303  0.71379407
  0.74825741  0.86583579  1.03608294  1.21247597  1.34562893  1.39673518
  1.34770715  1.20570205  1.00122908  0.7806941   0.59543563  0.48989467
  0.49164045  0.60528941  0.81148161  1.07086329  1.33212851  1.54230353
  1.65723169  1.65021588  1.51723797  1.27790776  0.97208585  0.65285079
  0.37720502  0.19611719  0.14560656  0.24042728  0.4712277   0.80570944
  1.19347409  1.57379212  1.88500433  2.07416191  2.10537784  1.96569534
  1.66758022  1.24772801  0.76232119  0.27944939 -0.13027224 -0.40357964
 -0.49358719 -0.37675623 -0.05687356  0.43451229  1.04122946  1.68888066
  2.29367454  2.77262774  3.0539410

In [31]:
# Stack up the plots
from bokeh.layouts import column
input_plot.title.text = "Integration of pulse to step"
show(column(input_plot, basis_plot, output_plot))

## C

In [33]:
c_times = np.linspace(0,8,100)
# Plot g0
g0_plot = figure(width=400, height=150, y_axis_label = r'\[g_0\]')
g0 = basis_function(c_times, 1, 0)
g0_plot.line(c_times, g0, line_width=2)
# Plot g1
g1_plot = figure(width=400, height=150, y_axis_label = r'\[g_1\]')
g1 = basis_function(c_times, 1, 1)
g1_plot.line(c_times, g1, line_width=2)
# Plot g2
g2_plot = figure(width=400, height=150, y_axis_label = r'\[g_2\]')
g2 = basis_function(c_times, 1, 2)
g2_plot.line(c_times, g2, line_width=2)

# Plot sums
sum_plot = figure(width=400, height=150, y_axis_label = r'sum', x_axis_label = r'Time (in units of \[\tau\])')
sum_plot.line(c_times, g0 + g1 + g2, line_width=2, legend_label=r'g0 + g1 + g2', color='blue')
sum_plot.line(c_times, g0 + 0.7 * g1 + 2 * g2, line_width=2, legend_label=r'g0 + 0.7g1 + 2g2', color='green')
sum_plot.line(np.linspace(0,2,10), np.ones(10), line_width=2, color='red', line_dash='dashed')

sum_plot.legend.location = "top_right"

show(column(g0_plot, g1_plot, g2_plot, sum_plot))

## D

In [34]:
# Varying inputs
impulse_Js = [0.5, 1, 2]

inputs_plot = figure(width=400, height=100, y_axis_label = 'Input', x_axis_label = 'Time (s)')
inputs_plot.yaxis.major_tick_line_color = None
inputs_plot.yaxis.minor_tick_line_color = None
inputs_plot.yaxis.major_label_text_font_size = '0pt'
for impulse_J in impulse_Js:
    # Compute the pulse
    input = pulse(t, t0, t1, impulse_J)

    # Plot the pulse
    inputs_plot.line(t, input, line_width=2, color='black')
show(inputs_plot)

In [37]:
# Compute the results for the varying inputs
outputs_plot = figure(width=400, height=300, y_axis_label = r'Output \[r_{out}\]', x_axis_label = 'Time (s)')
for impulse_J in impulse_Js:
    # Compute the pulse
    input = pulse(t, t0, t1, impulse_J)

    # Compute the weighted sum
    output = weighted_sum(t, tau, impulse_J * optimal_weights / J)

    # Plot the output
    outputs_plot.line(t, output, line_width=2)

show(outputs_plot)

In [38]:
# Stack up the plots
show(column(inputs_plot, outputs_plot))

## E

In [45]:
# Step input
def step(t, t0, A):
    return A * (t >= t0)

# Varying inputs
impulse_As = [0.5, 1, 2]

# Time Spacing
t = np.linspace(-0.4, 15, 1000)

inputs_plot = figure(width=400, height=100, y_axis_label = 'Input', x_axis_label = 'Time (s)')
inputs_plot.yaxis.major_tick_line_color = None
inputs_plot.yaxis.minor_tick_line_color = None
inputs_plot.yaxis.major_label_text_font_size = '0pt'
for A in impulse_As:
    # Compute the pulse
    input = step(t, t0, A)

    # Plot the pulse
    inputs_plot.line(t, input, line_width=2, color='black')
show(inputs_plot)


In [66]:
def dynamics(t, r, tau, weights, input):
    dr_dt = np.zeros_like(r)

    # first neuron receives the input
    dr_dt[0] = (-r[0] + input(t)) / tau

    # other neurons receive input from the previous neuron
    for n in range(1, len(r)-1):
        dr_dt[n] = (-r[n] + r[n-1]) / tau

    # last neuron receives input from all neurons and input weighted by the weights
    dr_dt[-1] = weights[0] * input(t) - r[-1]
    for n, w in enumerate(weights[1:]):
        dr_dt[-1] += w * r[n]

    dr_dt[-1] /= tau

    return dr_dt

output_plot = figure(width=400, height=300, y_axis_label = r'Output \[r_{out}\]', x_axis_label = 'Time (s)')
for A in impulse_As:
    # Parameters
    tau = 0.1 # seconds
    t0 = 0 # seconds
    input = lambda t: step(t, t0, A)
    weights = optimal_weights
    r0 = np.zeros(len(weights) + 1)

    params = (tau, weights, input)

    # Simulate the dynamics
    soln = scipy.integrate.solve_ivp(dynamics, (0, 15), r0, args=params)

    # Plot the output
    output_plot.line(soln.t, soln.y[-1], line_width=2)
show(output_plot)

In [68]:
# Stack up the plots
inputs_plot.title.text = "Integration of step to ramp"
show(column(inputs_plot, output_plot))

# Figure 7

In [180]:
# Autapse dynamics
def autapse_dynamics(t, r, tau, w, b, s, input):
    
    gE = w * (r + s * input(t)) + b

    f_gE = np.maximum(0, gE - 0.1)

    return (f_gE - r) / tau

# optimize readout
def readout(r, w_readout):
    return np.dot(w_readout, r)

def readout_objective(w_readout, r, t, t0, t1):
    r = r[:, (t >= t0) & (t <= t1)]
    output = readout(r, w_readout)
    return np.sum((output - 1)**2)

n_neurons = 100

tunings = [0.94, 0.995, 1, 1.02]

plots = []

for tuning in tunings:
    w = tuning * np.ones(n_neurons)
    b = 0.1 * np.ones(n_neurons)
    s = np.random.choice([-1, 1], size=n_neurons) * np.random.random(size=n_neurons)
    tau = 0.1
    input = lambda t: pulse(t, t0, 0.2, tau)

    args = (tau, w, b, s, input)

    soln = scipy.integrate.solve_ivp(autapse_dynamics, (0, 15), np.random.rand(n_neurons), args=args, t_eval=np.linspace(0, 15, 1000))
    autapse_plot = figure(width=300, height=200, y_axis_label = r'Neuronal Response', x_axis_label = 'Time (s)')
    for i in range(1, n_neurons, 7):
        autapse_plot.line(soln.t, soln.y[i], line_width=2)
    

    readout_weights = scipy.optimize.minimize(readout_objective, np.random.rand(n_neurons), args=(soln.y, soln.t, 0, 2), bounds=[(-5, 5) for _ in range(n_neurons)]).x

    plot_readout = figure(width=300, height=200, y_axis_label = r'Summed Output', x_axis_label = 'Time (s)', y_range=(0, 2))
    plot_readout.line(soln.t, readout(soln.y, readout_weights), line_width=2)
    plot_readout.line(np.linspace(0,2,10), np.ones(10) * 1.05, line_width=2, color='red', line_dash='dashed')
    plot_readout.line(np.linspace(0,2,10), np.ones(10) * 0.95, line_width=2, color='red', line_dash='dashed')
    
    plots.append((autapse_plot, plot_readout))

from bokeh.layouts import row
show(row(*[column(autapse, readout) for autapse, readout in plots]))

In [186]:
def ff_dynamics(t, r, tau, input, scaler):
    dr_dt = np.zeros_like(r)

    # first neuron receives the input
    dr_dt[0] = (-r[0] + input(t) * scaler) / tau

    # other neurons receive input from the previous neuron
    for n in range(1, len(r)):
        dr_dt[n] = (-r[n] + r[n-1] * scaler) / tau

    return dr_dt

# optimize readout
def readout(r, w_readout):
    return np.dot(w_readout, r)

def readout_objective(w_readout, r, t, t0, t1):
    r = r[:, (t >= t0) & (t <= t1)]
    output = readout(r, w_readout)
    return np.sum((output - 1)**2)

n_neurons = 100

tunings = [0.94, 0.995, 1, 1.02]

plots = []

for tuning in tunings:
    tau = 0.1
    input = lambda t: pulse(t, t0, 0.2, tau)

    args = (tau, input, tuning)

    soln = scipy.integrate.solve_ivp(ff_dynamics, (0, 15), np.zeros(n_neurons), args=args, t_eval=np.linspace(0, 15, 1000))
    ff_plot = figure(width=300, height=200, y_axis_label = r'Neuronal Response', x_axis_label = 'Time (s)')
    for i in range(1, n_neurons, 7):
        ff_plot.line(soln.t, soln.y[i], line_width=2)
    

    readout_weights = scipy.optimize.minimize(readout_objective, optimal_weights, args=(soln.y, soln.t, 0, 10), bounds=[(-5, 5) for _ in range(n_neurons)]).x

    plot_readout = figure(width=300, height=200, y_axis_label = r'Summed Output', x_axis_label = 'Time (s)', y_range=(0, 2))
    plot_readout.line(soln.t, readout(soln.y, readout_weights), line_width=2)
    plot_readout.line(np.linspace(0,2,10), np.ones(10) * 1.05, line_width=2, color='red', line_dash='dashed')
    plot_readout.line(np.linspace(0,2,10), np.ones(10) * 0.95, line_width=2, color='red', line_dash='dashed')
    
    plots.append((ff_plot, plot_readout))

from bokeh.layouts import row
show(row(*[column(ff, readout) for ff, readout in plots]))

# Lesioning

In [190]:
# Autapse dynamics + Lesion
def autapse_dynamics(t, r, tau, w, b, s, input, lesion_index = None):

    if lesion_index is not None:
        r[lesion_index] = 0
    
    gE = w * (r + s * input(t)) + b

    f_gE = np.maximum(0, gE - 0.1)

    dt = (f_gE - r) / tau

    if lesion_index is not None:
        dt[lesion_index] = 0

    return dt

# optimize readout
def readout(r, w_readout):
    return np.dot(w_readout, r)

def readout_objective(w_readout, r, t, t0, t1):
    r = r[:, (t >= t0) & (t <= t1)]
    output = readout(r, w_readout)
    return np.sum((output - 1)**2)

n_neurons = 100

lesions = [None, 0, 30, 60]

plots = []

for lesion in lesions:
    w = np.ones(n_neurons)
    b = 0.1 * np.ones(n_neurons)
    s = np.random.choice([-1, 1], size=n_neurons) * np.random.random(size=n_neurons)
    tau = 0.1
    input = lambda t: pulse(t, t0, 0.2, tau)

    args = (tau, w, b, s, input, lesion)

    soln = scipy.integrate.solve_ivp(autapse_dynamics, (0, 15), np.random.rand(n_neurons), args=args, t_eval=np.linspace(0, 15, 1000))
    autapse_plot = figure(width=300, height=200, y_axis_label = r'Neuronal Response', x_axis_label = 'Time (s)')
    for i in range(1, n_neurons, 7):
        autapse_plot.line(soln.t, soln.y[i], line_width=2)

    if lesion is not None:
        autapse_plot.title.text = f"Lesion at neuron {lesion}"
    

    readout_weights = scipy.optimize.minimize(readout_objective, np.random.rand(n_neurons), args=(soln.y, soln.t, 0, 2), bounds=[(-5, 5) for _ in range(n_neurons)]).x

    plot_readout = figure(width=300, height=200, y_axis_label = r'Summed Output', x_axis_label = 'Time (s)', y_range=(0, 2))
    plot_readout.line(soln.t, readout(soln.y, readout_weights), line_width=2)
    plot_readout.line(np.linspace(0,2,10), np.ones(10) * 1.05, line_width=2, color='red', line_dash='dashed')
    plot_readout.line(np.linspace(0,2,10), np.ones(10) * 0.95, line_width=2, color='red', line_dash='dashed')
    
    plots.append((autapse_plot, plot_readout))

from bokeh.layouts import row
show(row(*[column(autapse, readout) for autapse, readout in plots]))

In [191]:
def ff_dynamics(t, r, tau, input, scaler, lesion_index = None):
    if lesion_index is not None:
        r[lesion_index] = 0

    dr_dt = np.zeros_like(r)

    # first neuron receives the input
    dr_dt[0] = (-r[0] + input(t) * scaler) / tau

    # other neurons receive input from the previous neuron
    for n in range(1, len(r)):
        dr_dt[n] = (-r[n] + r[n-1] * scaler) / tau

    if lesion_index is not None:
        dr_dt[lesion_index] = 0

    return dr_dt

# optimize readout
def readout(r, w_readout):
    return np.dot(w_readout, r)

def readout_objective(w_readout, r, t, t0, t1):
    r = r[:, (t >= t0) & (t <= t1)]
    output = readout(r, w_readout)
    return np.sum((output - 1)**2)

n_neurons = 100

lesions = [None, 0, 30, 60]

plots = []

for lesion in lesions:
    tau = 0.1
    input = lambda t: pulse(t, t0, 0.2, tau)

    args = (tau, input, 1, lesion)

    soln = scipy.integrate.solve_ivp(ff_dynamics, (0, 15), np.zeros(n_neurons), args=args, t_eval=np.linspace(0, 15, 1000))
    ff_plot = figure(width=300, height=200, y_axis_label = r'Neuronal Response', x_axis_label = 'Time (s)')
    for i in range(1, n_neurons, 7):
        ff_plot.line(soln.t, soln.y[i], line_width=2)
    if lesion is not None:
        ff_plot.title.text = f"Lesion at neuron {lesion}"

    readout_weights = scipy.optimize.minimize(readout_objective, optimal_weights, args=(soln.y, soln.t, 0, 10), bounds=[(-5, 5) for _ in range(n_neurons)]).x

    plot_readout = figure(width=300, height=200, y_axis_label = r'Summed Output', x_axis_label = 'Time (s)', y_range=(0, 2))
    plot_readout.line(soln.t, readout(soln.y, readout_weights), line_width=2)
    plot_readout.line(np.linspace(0,2,10), np.ones(10) * 1.05, line_width=2, color='red', line_dash='dashed')
    plot_readout.line(np.linspace(0,2,10), np.ones(10) * 0.95, line_width=2, color='red', line_dash='dashed')
    
    plots.append((ff_plot, plot_readout))

from bokeh.layouts import row
show(row(*[column(ff, readout) for ff, readout in plots]))

In [201]:
def ff_dynamics_double(t, r, tau, input, lesion_index = None):
    if lesion_index is not None:
        r[lesion_index] = 0

    dr_dt = np.zeros_like(r)

    # first neuron receives the input
    dr_dt[0] = (-r[0] + input(t)) / tau

    # And second
    dr_dt[1] = (-r[1] + input(t)/2 + r[0]/2) / tau

    # other neurons receive input from the previous 2 neurons
    for n in range(2, len(r)):
        dr_dt[n] = (-r[n] + r[n-1]/2 + r[n-2]/2) / tau

    if lesion_index is not None:
        dr_dt[lesion_index] = 0

    return dr_dt

# optimize readout
def readout(r, w_readout):
    return np.dot(w_readout, r)

def readout_objective(w_readout, r, t, t0, t1):
    r = r[:, (t >= t0) & (t <= t1)]
    output = readout(r, w_readout)
    return np.sum((output - 1)**2)

n_neurons = 100

lesions = [None, 0, 30, 60]

plots = []

for lesion in lesions:
    tau = 0.1
    input = lambda t: pulse(t, t0, 0.2, tau)

    args = (tau, input, lesion)

    soln = scipy.integrate.solve_ivp(ff_dynamics_double, (0, 15), np.zeros(n_neurons), args=args, t_eval=np.linspace(0, 15, 1000))
    ff_plot = figure(width=300, height=200, y_axis_label = r'Neuronal Response', x_axis_label = 'Time (s)')
    for i in range(1, n_neurons, 7):
        ff_plot.line(soln.t, soln.y[i], line_width=2)
    if lesion is not None:
        ff_plot.title.text = f"Lesion at neuron {lesion}"

    readout_weights = scipy.optimize.minimize(readout_objective, optimal_weights, args=(soln.y, soln.t, 0, 10), bounds=[(-5, 5) for _ in range(n_neurons)]).x

    plot_readout = figure(width=300, height=200, y_axis_label = r'Summed Output', x_axis_label = 'Time (s)', y_range=(0, 2))
    plot_readout.line(soln.t, readout(soln.y, readout_weights), line_width=2)
    plot_readout.line(np.linspace(0,2,10), np.ones(10) * 1.05, line_width=2, color='red', line_dash='dashed')
    plot_readout.line(np.linspace(0,2,10), np.ones(10) * 0.95, line_width=2, color='red', line_dash='dashed')
    
    plots.append((ff_plot, plot_readout))

from bokeh.layouts import row
show(row(*[column(ff, readout) for ff, readout in plots]))

# Noise

In [196]:
# Autapse dynamics + Noise
def autapse_dynamics(t, r, tau, w, b, s, input, noise_std = 0):
    input_noise = noise_std * np.random.randn()
    # if input(t) > 0:
    #     input_noise = 0

    gE = w * (r + s * (input(t) + input_noise)) + b

    f_gE = np.maximum(0, gE - 0.1)

    dt = (f_gE - r) / tau

    return dt

# optimize readout
def readout(r, w_readout):
    return np.dot(w_readout, r)

def readout_objective(w_readout, r, t, t0, t1):
    r = r[:, (t >= t0) & (t <= t1)]
    output = readout(r, w_readout)
    return np.sum((output - 1)**2)

n_neurons = 100

noise_levels = [0, 0.01, 0.1, 0.5]

plots = []

w = np.ones(n_neurons)
b = 0.1 * np.ones(n_neurons)
s = np.random.choice([-1, 1], size=n_neurons) * np.random.random(size=n_neurons)
tau = 0.1
input = lambda t: pulse(t, t0, 0.2, tau)
r0 = np.random.rand(n_neurons)

for noise in noise_levels:

    args = (tau, w, b, s, input, noise)

    soln = scipy.integrate.solve_ivp(autapse_dynamics, (0, 15), r0, args=args, t_eval=np.linspace(0, 15, 1000))
    autapse_plot = figure(width=300, height=200, y_axis_label = r'Neuronal Response', x_axis_label = 'Time (s)')
    for i in range(1, n_neurons, 7):
        autapse_plot.line(soln.t, soln.y[i], line_width=2)

    if noise > 0:
        autapse_plot.title.text = f"Noise Level: {noise}"

    if noise == 0:
        readout_weights = scipy.optimize.minimize(readout_objective, np.random.rand(n_neurons), args=(soln.y, soln.t, 0, 2), bounds=[(-5, 5) for _ in range(n_neurons)]).x

    plot_readout = figure(width=300, height=200, y_axis_label = r'Summed Output', x_axis_label = 'Time (s)', y_range=(0, 2))
    plot_readout.line(soln.t, readout(soln.y, readout_weights), line_width=2)
    plot_readout.line(np.linspace(0,2,10), np.ones(10) * 1.05, line_width=2, color='red', line_dash='dashed')
    plot_readout.line(np.linspace(0,2,10), np.ones(10) * 0.95, line_width=2, color='red', line_dash='dashed')
    
    plots.append((autapse_plot, plot_readout))

from bokeh.layouts import row
show(row(*[column(autapse, readout) for autapse, readout in plots]))

In [ ]:
def ff_dynamics(t, r, tau, input, scaler, noise_std = 0):
    input_noise = noise_std * np.random.randn()

    dr_dt = np.zeros_like(r)

    # first neuron receives the input
    dr_dt[0] = (-r[0] + (input(t) + input_noise) * scaler) / tau

    # other neurons receive input from the previous neuron
    for n in range(1, len(r)):
        dr_dt[n] = (-r[n] + r[n-1] * scaler) / tau

    return dr_dt

# optimize readout
def readout(r, w_readout):
    return np.dot(w_readout, r)

def readout_objective(w_readout, r, t, t0, t1):
    r = r[:, (t >= t0) & (t <= t1)]
    output = readout(r, w_readout)
    return np.sum((output - 1)**2)

n_neurons = 100

noise_levels = [0, 0.01, 0.1, 0.5]

plots = []

tau = 0.1

for noise_level in noise_levels:
    input = lambda t: pulse(t, t0, 0.2, tau)

    args = (tau, input, 1, noise_level)

    soln = scipy.integrate.solve_ivp(ff_dynamics, (0, 15), np.zeros(n_neurons), args=args, t_eval=np.linspace(0, 15, 1000))
    ff_plot = figure(width=300, height=200, y_axis_label = r'Neuronal Response', x_axis_label = 'Time (s)')
    for i in range(1, n_neurons, 7):
        ff_plot.line(soln.t, soln.y[i], line_width=2)

    if noise_level > 0:
        ff_plot.title.text = f"Noise Level: {noise_level}"

    if noise_level == 0:
        readout_weights = scipy.optimize.minimize(readout_objective, optimal_weights, args=(soln.y, soln.t, 0, 10), bounds=[(-5, 5) for _ in range(n_neurons)]).x

    plot_readout = figure(width=300, height=200, y_axis_label = r'Summed Output', x_axis_label = 'Time (s)', y_range=(0, 2))
    plot_readout.line(soln.t, readout(soln.y, readout_weights), line_width=2)
    plot_readout.line(np.linspace(0,2,10), np.ones(10) * 1.05, line_width=2, color='red', line_dash='dashed')
    plot_readout.line(np.linspace(0,2,10), np.ones(10) * 0.95, line_width=2, color='red', line_dash='dashed')
    
    plots.append((ff_plot, plot_readout))

from bokeh.layouts import row
show(row(*[column(ff, readout) for ff, readout in plots]))

# Memory Capacity

In [232]:
import tqdm

In [271]:
n_neurons = 100

w = np.ones(n_neurons)
b = 0.1 * np.ones(n_neurons)
s = np.random.choice([-1, 1], size=n_neurons) * np.random.random(size=n_neurons)
tau = 0.1
r0 = np.random.rand(n_neurons)

input = lambda t: pulse(t, 0, 0.2, tau)

args = (tau, w, b, s, input, 0)

soln = scipy.integrate.solve_ivp(autapse_dynamics, (0, 10), r0, args=args, t_eval=np.linspace(0, 10, 1000))
readout_weights = scipy.optimize.minimize(readout_objective, np.random.rand(n_neurons), args=(soln.y, soln.t, 0, 10), bounds=[(-5, 5) for _ in range(n_neurons)]).x

In [327]:
stimuli = (np.random.random(size=1000)) * 10

inout_pairs = []

for noise in [0.01, 0.05]:
    resps = []
    for stim in tqdm.tqdm(stimuli):
        input = lambda t: pulse(t, 0, 0.2, tau * stim)

        args = (tau, w, b, s, input, noise)

        soln = scipy.integrate.solve_ivp(autapse_dynamics, (0, 10), r0, args=args)

        resp = readout(soln.y[:,-1], readout_weights)
        resps.append(resp)
    inout_pairs.append((stimuli, resps))

100%|██████████| 1000/1000 [01:42<00:00,  9.78it/s]


In [328]:
for stimuli, resps in inout_pairs:
    plot = figure(width=300, height=200, x_axis_label = "Stimulus", y_axis_label = "Response")
    plot.scatter(stimuli, resps, size=5)
    show(plot)

In [330]:
mi_plot = figure(width=400, height=300, x_axis_label = "Number of Bins", y_axis_label = "Mutual Information (bits)")
colors = ['blue', 'green']
for (stimuli, resps), color, noise in zip(inout_pairs, colors, [0.01, 0.05]):
    n_bins_vals = np.arange(1, 40)
    mi_vals = []

    for n_bins in n_bins_vals:
        stim_bins = np.linspace(stimuli.min(), stimuli.max(), n_bins + 1)
        resp_bins = np.linspace(min(resps), max(resps), n_bins + 1)

        dual = np.histogram2d(stimuli, resps, bins=[stim_bins, resp_bins], density=True)[0]

        # print("Dual histogram for n_bins =", n_bins, ":", dual)
        stim_hist = np.histogram(stimuli, bins=stim_bins, density=True)[0]
        resp_hist = np.histogram(resps, bins=resp_bins, density=True)[0]

        mi = 0
        for i in range(len(stim_bins)-1):
            for j in range(len(resp_bins)-1):
                if dual[i, j] > 0 and stim_hist[i] > 0 and resp_hist[j] > 0:
                    mi += dual[i, j] * (resp_bins[j+1] - resp_bins[j]) * (stim_bins[i+1] - stim_bins[i]) * np.log2(dual[i, j] / (stim_hist[i] * resp_hist[j]))

        mi_vals.append(mi)

    mi_plot.line(n_bins_vals, mi_vals, line_width=2, color=color, legend_label=f"Noise = {noise}")

mi_plot.legend.location = "bottom_right"
mi_plot.title.text = "Mutual Information, Autapse with Noise"
show(mi_plot)

In [315]:
n_neurons = 100

tau = 0.1

input = lambda t: pulse(t, 0, 0.2, tau)

args = (tau, input, 1, 0)

soln = scipy.integrate.solve_ivp(ff_dynamics, (0, 15), np.zeros(n_neurons), args=args, t_eval=np.linspace(0, 15, 1000))
readout_weights = scipy.optimize.minimize(readout_objective, optimal_weights, args=(soln.y, soln.t, 0, 10), bounds=[(-5, 5) for _ in range(n_neurons)]).x

In [322]:
stimuli = (np.random.random(size=200)) * 10

inout_pairs = []

for noise in [0.01, 0.05]:
    resps = []
    for stim in tqdm.tqdm(stimuli):
        input = lambda t: pulse(t, 0, 0.2, tau * stim)

        args = (tau, input, 1, noise)

        soln = scipy.integrate.solve_ivp(ff_dynamics, (0, 10), np.zeros(n_neurons), args=args)

        resp = readout(soln.y[:,-1], readout_weights)
        resps.append(resp)
    inout_pairs.append((stimuli, resps))

100%|██████████| 200/200 [08:07<00:00,  2.44s/it]


In [323]:
for stimuli, resps in inout_pairs:
    plot = figure(width=300, height=200, x_axis_label = "Stimulus", y_axis_label = "Response")
    plot.scatter(stimuli, resps, size=5)
    show(plot)

In [326]:
mi_plot = figure(width=400, height=300, x_axis_label = "Number of Bins", y_axis_label = "Mutual Information (bits)")
colors = ['blue', 'green']
for (stimuli, resps), color, noise in zip(inout_pairs, colors, [0.01, 0.05]):
    n_bins_vals = np.arange(1, 40)
    mi_vals = []

    for n_bins in n_bins_vals:
        stim_bins = np.linspace(stimuli.min(), stimuli.max(), n_bins + 1)
        resp_bins = np.linspace(min(resps), max(resps), n_bins + 1)

        dual = np.histogram2d(stimuli, resps, bins=[stim_bins, resp_bins], density=True)[0]

        # print("Dual histogram for n_bins =", n_bins, ":", dual)
        stim_hist = np.histogram(stimuli, bins=stim_bins, density=True)[0]
        resp_hist = np.histogram(resps, bins=resp_bins, density=True)[0]

        mi = 0
        for i in range(len(stim_bins)-1):
            for j in range(len(resp_bins)-1):
                if dual[i, j] > 0 and stim_hist[i] > 0 and resp_hist[j] > 0:
                    mi += dual[i, j] * (resp_bins[j+1] - resp_bins[j]) * (stim_bins[i+1] - stim_bins[i]) * np.log2(dual[i, j] / (stim_hist[i] * resp_hist[j]))

        mi_vals.append(mi)

    mi_plot.line(n_bins_vals, mi_vals, line_width=2, color=color, legend_label=f"Noise = {noise}")

mi_plot.legend.location = "bottom_right"
mi_plot.title.text = "Mutual Information, Feed Forward with Noise"
show(mi_plot)